# 01. 미분과 그라디언트

미분은 **입력이 조금 변할 때 출력이 얼마나 변하는지**를 보는 도구다.
로보틱스에서는 위치 오차, 에너지, 비용 함수가 관절각이나 상태에 대해 얼마나 민감한지 계산할 때 계속 등장한다.

$$\frac{df}{dx} = \lim_{h\to 0}\frac{f(x+h)-f(x)}{h}, \qquad \nabla f = \begin{bmatrix}\partial f/\partial x \\ \partial f/\partial y\end{bmatrix}$$

**로보틱스 연결:**
- 센서 오차 함수의 기울기 → 파라미터 보정 방향
- 경로 계획의 potential field → 장애물에서 멀어지는 방향
- 최적제어와 SLAM → 비용 함수의 gradient/Jacobian

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. 수치 미분과 해석 미분 비교

간단한 함수 $f(x)=\sin(x)+0.1x^2$ 에 대해 중앙차분으로 미분을 근사한다.

$$f'(x) \approx \frac{f(x+h)-f(x-h)}{2h}$$

In [ ]:
def f(x):
    return np.sin(x) + 0.1 * x**2

def df_analytical(x):
    return np.cos(x) + 0.2 * x

def df_numerical(x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

x = np.linspace(-6, 6, 400)
err = np.abs(df_analytical(x) - df_numerical(x))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(x, f(x), color='#534AB7', lw=2.5, label='f(x)')
axes[0].plot(x, df_analytical(x), color='#E85D24', lw=2, label="f'(x)")
axes[0].axhline(0, color='gray', lw=1)
axes[0].set_title('함수와 기울기')
axes[0].grid(alpha=0.25); axes[0].legend()

axes[1].semilogy(x, err + 1e-16, color='#1D9E75', lw=2)
axes[1].set_title('중앙차분 미분 오차')
axes[1].set_xlabel('x'); axes[1].set_ylabel('absolute error')
axes[1].grid(alpha=0.25)

plt.tight_layout()
plt.savefig('assets/01_derivatives_numeric.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'최대 오차: {err.max():.2e}')

## 2. 그라디언트는 가장 가파른 증가 방향

2차원 비용 함수:

$$J(x,y)=(x-1)^2 + 2(y+0.5)^2$$

그라디언트는 비용이 증가하는 방향이고, $-\nabla J$ 는 비용을 줄이는 방향이다.

In [ ]:
def cost(p):
    x, y = p
    return (x - 1.0)**2 + 2.0 * (y + 0.5)**2

def grad_cost(p):
    x, y = p
    return np.array([2 * (x - 1.0), 4 * (y + 0.5)])

xs = np.linspace(-2.5, 3.0, 120)
ys = np.linspace(-2.5, 2.0, 120)
X, Y = np.meshgrid(xs, ys)
Z = (X - 1.0)**2 + 2.0 * (Y + 0.5)**2

# gradient descent path
p = np.array([-2.0, 1.6])
path = [p.copy()]
alpha = 0.12
for _ in range(30):
    p = p - alpha * grad_cost(p)
    path.append(p.copy())
path = np.array(path)

fig, ax = plt.subplots(figsize=(8, 7))
cont = ax.contour(X, Y, Z, levels=25, cmap='viridis')
ax.clabel(cont, inline=True, fontsize=8)
ax.plot(path[:, 0], path[:, 1], 'o-', color='#E85D24', lw=2.5, markersize=4, label='gradient descent')
ax.plot(1.0, -0.5, '*', color='#1D9E75', markersize=16, label='minimum')

# vector field, sparse
sx = np.linspace(-2, 2.5, 9)
sy = np.linspace(-2, 1.5, 8)
SX, SY = np.meshgrid(sx, sy)
U = -2 * (SX - 1.0)
V = -4 * (SY + 0.5)
N = np.sqrt(U**2 + V**2) + 1e-9
ax.quiver(SX, SY, U/N, V/N, color='gray', alpha=0.45, label='-∇J direction')

ax.set_aspect('equal')
ax.set_title('비용 함수 지형과 하강 경로')
ax.grid(alpha=0.2); ax.legend()
plt.savefig('assets/01_gradient_descent.png', dpi=150, bbox_inches='tight')
plt.show()

print('마지막 위치:', path[-1].round(4), '비용:', cost(path[-1]).round(8))

## 3. Potential Field — 목표로 끌리고 장애물에서 밀림

로봇 경로 계획에서 자주 쓰는 직관:

$$U(q)=U_{att}(q)+U_{rep}(q)$$

로봇은 $-\nabla U(q)$ 방향으로 움직인다.

In [ ]:
goal = np.array([2.0, 1.5])
obstacles = [np.array([-0.4, 0.1]), np.array([0.9, -0.6])]

k_att = 1.0
k_rep = 0.12
rho0 = 0.8

def potential(q):
    u = 0.5 * k_att * np.sum((q - goal)**2)
    for obs in obstacles:
        rho = np.linalg.norm(q - obs)
        if rho < rho0:
            u += 0.5 * k_rep * (1/rho - 1/rho0)**2
    return u

def grad_potential(q, h=1e-4):
    g = np.zeros(2)
    for i in range(2):
        e = np.zeros(2); e[i] = h
        g[i] = (potential(q + e) - potential(q - e)) / (2*h)
    return g

xg = np.linspace(-2.2, 2.6, 100)
yg = np.linspace(-2.0, 2.1, 100)
X, Y = np.meshgrid(xg, yg)
Ufield = np.array([[potential(np.array([x, y])) for x in xg] for y in yg])
Ufield = np.clip(Ufield, 0, 8)

q = np.array([-1.8, -1.4])
traj = [q.copy()]
for _ in range(90):
    g = grad_potential(q)
    step = -0.08 * g / (np.linalg.norm(g) + 1e-9)
    q = q + step
    traj.append(q.copy())
    if np.linalg.norm(q - goal) < 0.08:
        break
traj = np.array(traj)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.contourf(X, Y, Ufield, levels=30, cmap='magma_r')
plt.colorbar(im, ax=ax, label='potential')
ax.plot(traj[:,0], traj[:,1], 'o-', color='white', lw=2, markersize=3, label='robot path')
ax.plot(*goal, '*', color='#1D9E75', markersize=18, label='goal')
for obs in obstacles:
    circle = plt.Circle(obs, rho0, color='#E85D24', alpha=0.18)
    ax.add_patch(circle)
    ax.plot(*obs, 'x', color='#E85D24', markersize=12, mew=3)
ax.set_aspect('equal')
ax.set_title('Potential field 기반 경로 생성')
ax.legend(loc='upper left')
plt.savefig('assets/01_potential_field.png', dpi=150, bbox_inches='tight')
plt.show()

## 요약

| 개념 | 의미 | 로보틱스 활용 |
|------|------|---------------|
| 미분 | 작은 변화에 대한 민감도 | 센서 모델, 속도, 선형화 |
| 그라디언트 | 가장 가파른 증가 방향 | 최적화, 경로 계획 |
| 경사하강법 | $x \leftarrow x-\alpha\nabla J$ | 캘리브레이션, SLAM, 제어 파라미터 튜닝 |
| Potential field | 목표/장애물을 비용으로 모델링 | 간단한 local path planning |